In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans, MeanShift
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.mixture import GaussianMixture

In [2]:
# 현재 Notebook이 training 폴더에서 실행되는 경우
# 프로젝트 최상위 폴더를 찾습니다.

if Path.cwd().name == "training":
    PROJECT_ROOT = Path.cwd().parent

else:
    PROJECT_ROOT = Path.cwd()


DATA_PATH = PROJECT_ROOT / "data" / "phishing_url_features.csv"

MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(exist_ok=True)

print("프로젝트 경로:", PROJECT_ROOT)
print("데이터 경로:", DATA_PATH)
print("모델 저장 경로:", MODELS_DIR)

프로젝트 경로: /Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지
데이터 경로: /Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/data/phishing_url_features.csv
모델 저장 경로: /Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/models


In [3]:
data = pd.read_csv(DATA_PATH)

X = data.drop(["url", "label"], axis=1)

y = data["label"]


print("전체 데이터 크기:", data.shape)
print("Feature 개수:", X.shape[1])

display(data.head())

전체 데이터 크기: (22000, 17)
Feature 개수: 15


,having_ip_addr,url_length,shortening_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,domain_registration_length,Favicon,Port,HTTPS_token,age_of_domain,dns_record,count_Redirection,Disabling_Right_Click,url,label
0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,https://www.google.com,0
1,1,1,1,1,1,1,1,1,-1,1,1,1,1,1,1,https://www.facebook.com,0
2,1,1,1,1,1,1,1,1,-1,1,1,1,1,1,1,https://www.microsoft.com,0
3,1,1,1,1,1,1,1,1,1,1,1,-1,1,1,1,https://www.apple.com,0
4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,https://www.cisco.com,0


In [4]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)

kmeans_result = kmeans.fit_predict(X)


print("클러스터 분포:", np.unique(kmeans_result, return_counts=True))


label_df = pd.DataFrame({"label": y, "cluster": kmeans_result})


display(label_df.groupby("label")["cluster"].value_counts())


# 실제 label과 cluster 번호를 연결
cluster_label_mapping = label_df.groupby("cluster")["label"].mean().to_dict()


cluster_to_label = {
    cluster: (1 if mean_label > 0.5 else 0)
    for cluster, mean_label in cluster_label_mapping.items()
}


kmeans_pred = [cluster_to_label[cluster] for cluster in kmeans_result]


kmeans_accuracy = accuracy_score(y, kmeans_pred) * 100


print(f"K-Means Accuracy: {kmeans_accuracy:.2f}%")


joblib.dump(kmeans, MODELS_DIR / "kmeans.pkl")

클러스터 분포: (array([0, 1], dtype=int32), array([13320,  8680]))


label  cluster
0      0          13249
       1           1751
1      1           6929
       0             71
Name: count, dtype: int64

K-Means Accuracy: 91.72%


['/Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/models/kmeans.pkl']

In [5]:
gmm = GaussianMixture(n_components=2, n_init=10, random_state=42)

gmm.fit(X)

gmm_result = gmm.predict(X)


print("클러스터 분포:", np.unique(gmm_result, return_counts=True))


label_df = pd.DataFrame({"label": y, "cluster": gmm_result})


display(label_df.groupby("label")["cluster"].value_counts())


cluster_label_mapping = label_df.groupby("cluster")["label"].mean().to_dict()


cluster_to_label = {
    cluster: (1 if mean_label > 0.5 else 0)
    for cluster, mean_label in cluster_label_mapping.items()
}


gmm_pred = [cluster_to_label[cluster] for cluster in gmm_result]


gmm_accuracy = accuracy_score(y, gmm_pred) * 100


print(f"GMM Accuracy: {gmm_accuracy:.2f}%")


joblib.dump(gmm, MODELS_DIR / "gmm.pkl")

클러스터 분포: (array([0, 1]), array([20528,  1472]))


label  cluster
0      0          14990
       1             10
1      0           5538
       1           1462
Name: count, dtype: int64

GMM Accuracy: 74.78%


['/Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/models/gmm.pkl']

In [6]:
meanshift = MeanShift(bandwidth=2.27)


meanshift_result = meanshift.fit_predict(X)


print("클러스터 분포:", np.unique(meanshift_result, return_counts=True))


label_df = pd.DataFrame({"label": y, "cluster": meanshift_result})


display(label_df.groupby("label")["cluster"].value_counts())


meanshift_accuracy = accuracy_score(y, meanshift_result) * 100


print(f"MeanShift Accuracy: {meanshift_accuracy:.2f}%")


joblib.dump(meanshift, MODELS_DIR / "meanshift.pkl")

클러스터 분포: (array([0, 1]), array([17436,  4564]))


label  cluster
0      0          14377
       1            623
1      1           3941
       0           3059
Name: count, dtype: int64

MeanShift Accuracy: 83.26%


['/Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/models/meanshift.pkl']

In [7]:
agglomerative = AgglomerativeClustering(n_clusters=2)


agglomerative_labels = agglomerative.fit_predict(X)


print("클러스터 분포:", np.unique(agglomerative_labels, return_counts=True))


label_df = pd.DataFrame({"label": y, "cluster": agglomerative_labels})


display(label_df.groupby("label")["cluster"].value_counts())


agglomerative_accuracy = accuracy_score(y, agglomerative_labels) * 100


print(f"Agglomerative Accuracy: {agglomerative_accuracy:.2f}%")


joblib.dump(agglomerative, MODELS_DIR / "agglomerative.pkl")


# Agglomerative는 새로운 데이터에 predict()가 없기 때문에
# 학습 당시 군집 레이블도 따로 저장
joblib.dump(agglomerative_labels, MODELS_DIR / "agglomerative_labels.pkl")

클러스터 분포: (array([0, 1]), array([12478,  9522]))


label  cluster
0      0          12452
       1           2548
1      1           6974
       0             26
Name: count, dtype: int64

Agglomerative Accuracy: 88.30%


['/Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/models/agglomerative_labels.pkl']

In [8]:
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)


random_forest.fit(X, y)


random_forest_pred = random_forest.predict(X)


random_forest_accuracy = accuracy_score(y, random_forest_pred) * 100


print(f"Random Forest Accuracy: {random_forest_accuracy:.2f}%")


joblib.dump(random_forest, MODELS_DIR / "random_forest.pkl")

Random Forest Accuracy: 96.91%


['/Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/models/random_forest.pkl']

In [9]:
logistic_regression = LogisticRegression(random_state=42, max_iter=1000)


logistic_regression.fit(X, y)


logistic_regression_pred = logistic_regression.predict(X)


logistic_regression_accuracy = accuracy_score(y, logistic_regression_pred) * 100


print(f"Logistic Regression Accuracy: {logistic_regression_accuracy:.2f}%")


joblib.dump(logistic_regression, MODELS_DIR / "logistic_regression.pkl")

Logistic Regression Accuracy: 95.36%


['/Users/sjs/Documents/졸업작품/졸업작품/비지도학습기반악성URL탐지/models/logistic_regression.pkl']

In [10]:
results = pd.DataFrame(
    {
        "Model": [
            "K-Means",
            "GMM",
            "MeanShift",
            "Agglomerative Clustering",
            "Random Forest",
            "Logistic Regression",
        ],
        "Type": [
            "Unsupervised",
            "Unsupervised",
            "Unsupervised",
            "Unsupervised",
            "Supervised",
            "Supervised",
        ],
        "Accuracy (%)": [
            kmeans_accuracy,
            gmm_accuracy,
            meanshift_accuracy,
            agglomerative_accuracy,
            random_forest_accuracy,
            logistic_regression_accuracy,
        ],
    }
)


results = results.sort_values("Accuracy (%)", ascending=False)


display(results)

,Model,Type,Accuracy (%)
4,Random Forest,Supervised,96.913636
5,Logistic Regression,Supervised,95.359091
0,K-Means,Unsupervised,91.718182
3,Agglomerative Clustering,Unsupervised,88.300000
2,MeanShift,Unsupervised,83.263636
1,GMM,Unsupervised,74.781818


In [11]:
print("저장된 모델")

for model_path in sorted(MODELS_DIR.glob("*.pkl")):
    print("-", model_path.name)

저장된 모델
- agglomerative.pkl
- agglomerative_labels.pkl
- dbscan.pkl
- gmm.pkl
- kmeans.pkl
- logistic_regression.pkl
- meanshift.pkl
- random_forest.pkl
